In [35]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [36]:
# Helper functions
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    if tools:
        params["tools"] = tools

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

In [37]:
from datetime import datetime
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current date and time formatted according to the specified format string. Useful when you need to know the current date, time, or both in a specific format.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A strftime-compatible format string that controls how the datetime is returned. Defaults to '%Y-%m-%d %H:%M:%S' (e.g. '2025-01-30 14:35:00'). Common directives: %Y (4-digit year), %m (month), %d (day), %H (24h hour), %M (minute), %S (second). Must not be empty."
      }
    },
    "required": []
  }
})

In [38]:
import json

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)

def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type =="tool_use"
    ]
    tool_result_blocks = []

    for tool_request in tool_requests:
                try:
                    tool_output = run_tool(tool_request.name, tool_request.input)
                    tool_result_block = {
                        "type": "tool_result",
                        "tool_use_id":tool_request.id,
                        "content": json.dumps(tool_output),
                        "is_error": False
                    }
                except Exception as e:
                    tool_result_block = {
                        "type": "tool_result",
                        "tool_use_id":tool_request.id,
                        "content": f"Error: {e}",
                        "is_error": True
                    }

                tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [39]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [40]:
messages = []

add_user_message(
    messages,
    "What's the current time in HH:MM format? Also, what is the current time in SS format?"
)
run_conversation(messages)



The current time is:
- **HH:MM format**: 06:58
- **SS format (seconds)**: 01


[{'role': 'user',
  'content': "What's the current time in HH:MM format? Also, what is the current time in SS format?"},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_017K4hsa3BUsXcHJF68V9B5V', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type='tool_use'),
   ToolUseBlock(id='toolu_01A1qZhhoRaGXPksqSMw7PvR', caller=DirectCaller(type='direct'), input={'date_format': '%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_017K4hsa3BUsXcHJF68V9B5V',
    'content': '"06:58"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01A1qZhhoRaGXPksqSMw7PvR',
    'content': '"01"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time is:\n- **HH:MM format**: 06:58\n- **SS format (seconds)**: 01', type='text')]}]